# GCI Global 2026 Competition — Home Credit Default Risk
Pipeline completo reproducible de Feature Engineering, Entrenamiento Multi-Modelo y Ensamble.

## Módulo: `src/features.py`

In [ ]:
"""
Feature engineering pipeline - Home Credit Default Risk (competitive sprint)
============================================================================

Leak-free feature matrix shared by train and test (built on the concatenated
frame so statistics are computed identically for both):

- Cleaning: DAYS_EMPLOYED == 365243 -> NaN + DAYS_EMPLOYED_ANOM flag
- Financial ratios: CREDIT_TO_ANNUITY_RATIO (loan term in months),
  ANNUITY_TO_INCOME_RATIO, CREDIT_TO_INCOME_RATIO, CREDIT_TO_GOODS_RATIO,
  CREDIT_GOODS_DIFF (over-financing), per-person / per-child income
- EXT_SOURCE block: mean/std/min/max/sum/median, product (raw + imputed),
  pairwise diffs and safe ratios, weighted combination, squares, sqrts,
  exact null counts and per-source ISNA indicators
- Age / employment: years, employment-to-birth ratios, car-to-birth/employed,
  phone/id publish to birth ratios
- Bureau request SUM and MAX over AMT_REQ_CREDIT_BUREAU_*
- Categorical interactions: GENDER_EDU, INCOME_ORG, CONTRACT_GENDER
- Frequency encoding (normalized counts fit on train+test) for all
  categorical columns and interactions
- Group aggregations (mean, std, val-mean diff, val/(mean+eps) ratio) of
  EXT_SOURCE_MEAN/2/3, AMT_CREDIT, AMT_INCOME_TOTAL, CREDIT_TO_ANNUITY_RATIO,
  DAYS_BIRTH grouped by ORGANIZATION_TYPE, OCCUPATION_TYPE,
  NAME_EDUCATION_TYPE, NAME_INCOME_TYPE, GENDER_EDU
- One-hot encoding for the 8 base categorical columns
- Null count per row

Smoothed target encoding (m=20) is applied STRICTLY OUT-OF-FOLD inside the
training folds (see fit_smoothed_target_encoding below); it never enters
build_features, so no target information leaks into the static matrix.

All numeric outputs are downcast to float32 / int32.
"""

import gc

import numpy as np
import pandas as pd

# Original categorical columns (kept as strings for OOF target encoding)
CAT_COLUMNS = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
]

# Categorical columns used for frequency encoding (base + interactions)
FE_CAT_COLUMNS = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "GENDER_EDU",
    "INCOME_ORG",
    "CONTRACT_GENDER",
]

# Categorical columns receiving out-of-fold smoothed target encoding (m=20)
TARGET_ENCODED_COLS = [
    "ORGANIZATION_TYPE",
    "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE",
    "GENDER_EDU",
]

GROUP_COLS = [
    "ORGANIZATION_TYPE",
    "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE",
    "GENDER_EDU",
]

GROUP_AGG_VARS = [
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_CREDIT",
    "AMT_INCOME_TOTAL",
    "CREDIT_TO_ANNUITY_RATIO",
    "DAYS_BIRTH",
]

EXT_SOURCES = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

_MISSING = "__NA__"
_EPS = 1e-5


def _to_str_col(s):
    """Object column with NaN replaced by the explicit missing token."""
    return s.astype("object").where(s.notna(), _MISSING)


def _fill_cat(df, cols):
    """Replace NaNs in categorical columns with an explicit missing token."""
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = _to_str_col(out[col])
    return out


def _safe_div(num, den, eps=_EPS):
    return num / (den + eps)


def build_features(train_df, test_df):
    """
    Build the leak-free feature matrix shared by train and test.

    Parameters
    ----------
    train_df, test_df : pd.DataFrame
        Raw frames (already sorted by SK_ID_CURR upstream).

    Returns
    -------
    train_proc, test_proc : pd.DataFrame
        Aligned feature frames. train_proc keeps the TARGET column;
        test_proc has no TARGET. Column order is identical in both.
    """
    df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    df = df.astype({c: np.float32 for c in df.select_dtypes(include="float64").columns})

    # --- 0. Cleaning --------------------------------------------------------
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype("int32")
    df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

    income = df["AMT_INCOME_TOTAL"]
    credit = df["AMT_CREDIT"]
    annuity = df["AMT_ANNUITY"]
    goods = df["AMT_GOODS_PRICE"]

    # --- 1. Financial ratios -------------------------------------------------
    # Loan term in months - the #1 domain feature of Home Credit
    df["CREDIT_TO_ANNUITY_RATIO"] = _safe_div(credit, annuity)
    df["ANNUITY_TO_INCOME_RATIO"] = _safe_div(annuity, income)
    df["CREDIT_TO_INCOME_RATIO"] = _safe_div(credit, income)
    df["CREDIT_TO_GOODS_RATIO"] = _safe_div(credit, goods)
    df["CREDIT_GOODS_DIFF"] = (credit - goods).astype("float32")
    df["INCOME_PER_PERSON"] = _safe_div(income, df["CNT_FAM_MEMBERS"])
    df["INCOME_PER_CHILD"] = income / (1 + df["CNT_CHILDREN"])
    df["CHILDREN_RATIO"] = _safe_div(df["CNT_CHILDREN"], df["CNT_FAM_MEMBERS"])

    # --- 2. EXT_SOURCE block (maximum priority) -------------------------------
    ext = df[EXT_SOURCES]
    ext_imp = ext.fillna(0.0)
    df["EXT_SOURCE_MEAN"] = ext.mean(axis=1)
    df["EXT_SOURCE_STD"] = ext.std(axis=1)
    df["EXT_SOURCE_MIN"] = ext.min(axis=1)
    df["EXT_SOURCE_MAX"] = ext.max(axis=1)
    df["EXT_SOURCE_SUM"] = ext_imp.sum(axis=1)
    df["EXT_SOURCE_PROD"] = (ext_imp[EXT_SOURCES[0]] * ext_imp[EXT_SOURCES[1]] * ext_imp[EXT_SOURCES[2]]).astype("float32")
    df["EXT_SOURCES_RANGE"] = (ext.max(axis=1) - ext.min(axis=1)).astype("float32")
    df["EXT_SOURCE_1_MINUS_2"] = (ext[EXT_SOURCES[0]] - ext[EXT_SOURCES[1]]).astype("float32")
    df["EXT_SOURCE_2_MINUS_3"] = (ext[EXT_SOURCES[1]] - ext[EXT_SOURCES[2]]).astype("float32")
    df["EXT_SOURCE_1_MINUS_3"] = (ext[EXT_SOURCES[0]] - ext[EXT_SOURCES[2]]).astype("float32")
    df["EXT_1_OVER_2"] = _safe_div(ext[EXT_SOURCES[0]], ext[EXT_SOURCES[1]])
    df["EXT_2_OVER_3"] = _safe_div(ext[EXT_SOURCES[1]], ext[EXT_SOURCES[2]])
    df["EXT_SOURCE_WEIGHTED"] = (
        ext[EXT_SOURCES[0]] * 2 + ext[EXT_SOURCES[1]] * 3 + ext[EXT_SOURCES[2]] * 4
    )
    df["EXT_SOURCE_1_SQ"] = ext[EXT_SOURCES[0]] ** 2
    df["EXT_SOURCE_2_SQ"] = ext[EXT_SOURCES[1]] ** 2
    df["EXT_SOURCE_3_SQ"] = ext[EXT_SOURCES[2]] ** 2
    df["EXT_SOURCE_1_SQRT"] = np.sqrt(ext[EXT_SOURCES[0]].fillna(0.0).clip(lower=0.0))
    df["EXT_SOURCE_2_SQRT"] = np.sqrt(ext[EXT_SOURCES[1]].fillna(0.0).clip(lower=0.0))
    df["EXT_SOURCE_3_SQRT"] = np.sqrt(ext[EXT_SOURCES[2]].fillna(0.0).clip(lower=0.0))
    df["EXT_SOURCE_NA_COUNT"] = ext.isna().sum(axis=1).astype("int32")
    df["EXT_1_ISNA"] = ext[EXT_SOURCES[0]].isna().astype("int32")
    df["EXT_2_ISNA"] = ext[EXT_SOURCES[1]].isna().astype("int32")
    df["EXT_3_ISNA"] = ext[EXT_SOURCES[2]].isna().astype("int32")

    # --- 3. Age / employment / car / phone ------------------------------------
    age = (-df["DAYS_BIRTH"] / 365.0).astype("float32")
    df["AGE_YEARS"] = age
    df["AGE_BIN"] = pd.cut(
        age, bins=[0, 25, 35, 45, 55, 120], labels=False
    ).astype("float32")
    emp_years = (df["DAYS_EMPLOYED"] / 365.0).astype("float32")
    df["DAYS_EMPLOYED_YEARS"] = emp_years
    df["EMPLOYED_TO_BIRTH_RATIO"] = _safe_div(df["DAYS_EMPLOYED"], df["DAYS_BIRTH"])
    df["DAYS_EMPLOYED_DIFF"] = (df["DAYS_BIRTH"] - df["DAYS_EMPLOYED"]).astype("float32")
    df["CAR_TO_BIRTH_RATIO"] = _safe_div(df["OWN_CAR_AGE"], age)
    df["CAR_TO_EMPLOYED_RATIO"] = _safe_div(df["OWN_CAR_AGE"], emp_years)
    df["PHONE_TO_BIRTH_RATIO"] = _safe_div(df["DAYS_LAST_PHONE_CHANGE"], df["DAYS_BIRTH"])
    df["ID_TO_BIRTH_RATIO"] = _safe_div(df["DAYS_ID_PUBLISH"], df["DAYS_BIRTH"])

    # --- 4. Bureau request stats ----------------------------------------------
    bureau_cols = [c for c in df.columns if c.startswith("AMT_REQ_CREDIT_BUREAU")]
    bureau = df[bureau_cols]
    df["BUREAU_REQ_SUM"] = bureau.sum(axis=1).astype("float32")
    df["BUREAU_REQ_MAX"] = bureau.max(axis=1).astype("float32")

    # --- 5. Null count (over raw columns, before encodings) --------------------
    df["NULL_COUNT"] = df.isna().sum(axis=1).astype("int32")

    # --- 6. Categorical interactions -------------------------------------------
    filled = _fill_cat(df, CAT_COLUMNS)
    df["GENDER_EDU"] = filled["CODE_GENDER"] + "_" + filled["NAME_EDUCATION_TYPE"]
    df["INCOME_ORG"] = filled["NAME_INCOME_TYPE"] + "_" + filled["ORGANIZATION_TYPE"]
    df["CONTRACT_GENDER"] = filled["NAME_CONTRACT_TYPE"] + "_" + filled["CODE_GENDER"]

    # --- 7. Frequency encoding (fit on train+test, non-target statistics) -------
    for col in FE_CAT_COLUMNS:
        vc = df[col].value_counts(normalize=True)
        df[f"FREQ_{col}"] = df[col].map(vc).fillna(0.0).astype("float32")

    # --- 8. Group aggregations (fit on train+test, non-target statistics) -------
    group_cols_dict = {}
    for col in GROUP_COLS:
        grp = df.groupby(col)
        for var in GROUP_AGG_VARS:
            gmean = grp[var].transform("mean")
            gstd = grp[var].transform("std")
            group_cols_dict[f"G_{col}_{var}_MEAN"] = gmean.astype("float32")
            group_cols_dict[f"G_{col}_{var}_STD"] = gstd.astype("float32")
            group_cols_dict[f"G_{col}_{var}_DIFF"] = (df[var] - gmean).astype("float32")
            group_cols_dict[f"G_{col}_{var}_RATIO"] = _safe_div(df[var], gmean)

    df = pd.concat([df, pd.DataFrame(group_cols_dict, index=df.index)], axis=1)
    del group_cols_dict

    # --- 9. One-hot encoding ------------------------------------------------------
    dummies = pd.get_dummies(filled[CAT_COLUMNS], prefix=CAT_COLUMNS)
    dummies.columns = [sanitize_name(c) for c in dummies.columns]
    dummies = dummies.astype("float32")
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(columns=FE_CAT_COLUMNS)
    del dummies, filled

    # --- 10. Split back and downcast -----------------------------------------------
    mask = df["TARGET"].notna()
    train_proc = df[mask].copy()
    test_proc = df[~mask].drop(columns=["TARGET"]).copy()
    del df

    # Replace infinities with NaN (they can happen in ratios with 0 denom)
    for frame in (train_proc, test_proc):
        for col in frame.select_dtypes(include=["float64", "float32"]).columns:
            frame[col] = frame[col].replace([np.inf, -np.inf], np.nan)
        for col in frame.select_dtypes(include=["float64"]).columns:
            frame[col] = frame[col].astype("float32")
        for col in frame.select_dtypes(include=["int64"]).columns:
            frame[col] = frame[col].astype("int32")

    gc.collect()
    return train_proc, test_proc


def sanitize_name(name):
    """Make a column name safe (no special chars, no leading digit)."""
    import re

    sanitized = re.sub(r"[^a-zA-Z0-9_]+", "_", str(name)).strip("_")
    if sanitized and sanitized[0].isdigit():
        sanitized = "col_" + sanitized
    return sanitized if sanitized else "unnamed_col"


def get_target_encoding_cats(train_df, test_df, cols=None):
    """
    Build the NaN-filled categorical frames used for OOF target encoding.

    Returns (train_cats, test_cats, col_names) where both frames have the
    same column order and the same row order as the raw inputs.
    """
    cols = cols if cols is not None else TARGET_ENCODED_COLS

    train_cats = _fill_cat(train_df, CAT_COLUMNS)
    test_cats = _fill_cat(test_df, CAT_COLUMNS)
    train_cats["GENDER_EDU"] = (
        train_cats["CODE_GENDER"] + "_" + train_cats["NAME_EDUCATION_TYPE"]
    )
    test_cats["GENDER_EDU"] = (
        test_cats["CODE_GENDER"] + "_" + test_cats["NAME_EDUCATION_TYPE"]
    )

    order = [c for c in cols if c in train_cats.columns]
    return (
        train_cats[order].reset_index(drop=True),
        test_cats[order].reset_index(drop=True),
        order,
    )


def fit_smoothed_target_encoding(y, cat_values, smoothing=20):
    """
    Compute smoothed target encoding statistics on ONE fold's training part.

    encoding = (n_pos + m * g) / (n + m)

    where n_pos/n are the per-category positives/count in the fit data,
    g is the global positive rate of the fit data, and m is the smoothing
    strength pulling rare categories toward the global rate.

    Parameters
    ----------
    y : array-like, shape (n,)
        Binary targets of the training part of the fold.
    cat_values : array-like, shape (n,)
        Categorical values (string token for missing) of the training part.

    Returns
    -------
    (stats, global_rate) where stats is dict {category: (n, npos)}.
    """
    y_arr = np.asarray(y, dtype=np.float64)
    cats = np.asarray(cat_values)
    global_rate = float(y_arr.mean()) if len(y_arr) else 0.5
    df = pd.DataFrame({"cat": cats, "y": y_arr})
    grouped = df.groupby("cat")["y"]
    counts = grouped.size()
    sums = grouped.sum()
    stats = {c: (int(counts[c]), float(sums[c])) for c in counts.index}
    return stats, global_rate


def transform_target_encoding(stats, global_rate, cat_values, smoothing=20):
    """
    Apply smoothed target encoding using statistics fit on a fold's
    training part (out-of-fold by construction). Unknown categories fall
    back to the global positive rate.
    """
    cats = np.asarray(cat_values, dtype=object)
    # Vectorize lookup via a dict-backed Series
    keys = np.array(list(stats.keys()), dtype=object)
    n_arr = np.array([stats[k][0] for k in keys], dtype=np.float64)
    npos_arr = np.array([stats[k][1] for k in keys], dtype=np.float64)
    lookup = pd.Series(npos_arr, index=pd.Series(keys)).to_dict()
    n_lookup = pd.Series(n_arr, index=pd.Series(keys)).to_dict()

    n = np.array([n_lookup.get(c, 0.0) for c in cats], dtype=np.float64)
    npos = np.array([lookup.get(c, 0.0) for c in cats], dtype=np.float64)
    enc = (npos + smoothing * global_rate) / (n + smoothing)
    unknown = n == 0
    enc[unknown] = global_rate
    return enc.astype(np.float32)


def get_oof_target_encoder(smoothing=20):
    """
    Return a helper that produces out-of-fold smoothed target encodings.

    Usage inside a training fold (skf indices tr/va, raw cat frames):
        oof_te[va] = te(cat_train_va, y[tr], cat_raw_va)
        test_te += te(cat_raw, y[tr], cat_raw_te)/nfold
    """
    def te(cat_fit, y_fit, cat_target):
        stats, g = fit_smoothed_target_encoding(y_fit, cat_fit, smoothing=smoothing)
        return transform_target_encoding(stats, g, cat_target, smoothing=smoothing)

    return te


## Módulo: `src/models.py`

In [ ]:
"""
Model training functions - Home Credit Default Risk (competitive sprint)
========================================================================

All models are trained with the same protocol:

- StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- n_jobs <= 4, float32 features
- Early stopping whenever the backend supports it
- Returns (oof_preds, test_preds, auc)

Competitive lineup (3 GBDT families, multi-seed averaged):
- train_lgbm_competitive : LightGBM (lr 0.02, leaves 31, early stop 150)
- train_xgb_competitive  : XGBoost hist (lr 0.02, depth 5, early stop 150)
- train_catboost_competitive : CatBoost (depth 6, l2 10, early stop 150)
- SEEDS = [42, 2024, 777]; OOF and test predictions are averaged
  per family across the 3 seeds.

Legacy helpers kept for run_pipeline.py:
- train_lgbm, train_xgb, train_catboost, train_lr, train_et
"""

import gc

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

SEED = 42
SEEDS = [42, 2024, 777]
N_JOBS = 4
N_FOLDS = 5
EARLY_STOP = 150


def _make_cv(nfold):
    return StratifiedKFold(n_splits=nfold, shuffle=True, random_state=SEED)


# ---------------------------------------------------------------------------
# Competitive GBDT trainers (multi-seed aware)
# ---------------------------------------------------------------------------

def _lgbm_params(seed):
    return {
        "n_estimators": 4000,
        "learning_rate": 0.02,
        "num_leaves": 31,
        "max_depth": 7,
        "min_child_samples": 50,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.6,
        "reg_alpha": 5.0,
        "reg_lambda": 10.0,
        "n_jobs": N_JOBS,
        "random_state": seed,
        "verbosity": -1,
        "force_col_wise": True,
    }


def _xgb_params(seed):
    return {
        "n_estimators": 4000,
        "learning_rate": 0.02,
        "max_depth": 5,
        "min_child_weight": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.6,
        "reg_alpha": 5.0,
        "reg_lambda": 10.0,
        "n_jobs": N_JOBS,
        "random_state": seed,
        "tree_method": "hist",
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "early_stopping_rounds": EARLY_STOP,
    }


def _cb_params(seed):
    return {
        "iterations": 4500,
        "learning_rate": 0.025,
        "depth": 6,
        "l2_leaf_reg": 10.0,
        "subsample": 0.8,
        "border_count": 254,
        "random_seed": seed,
        "thread_count": N_JOBS,
        "task_type": "CPU",
        "verbose": 0,
        "allow_writing_files": False,
    }


def train_lgbm_competitive(
    X, y, X_test, nfold=N_FOLDS, seeds=None, feature_names=None
):
    """
    LightGBM with multi-seed averaging.

    Each seed runs the full 5-fold OOF; OOF and test predictions are
    averaged across seeds per sample.

    Returns (oof, test, auc, fold_best_iters, feature_importance).
    """
    seeds = seeds if seeds is not None else SEEDS
    skf = _make_cv(nfold)
    splits = list(skf.split(X, y))

    oof_total = np.zeros(len(X), dtype=np.float32)
    test_total = np.zeros(len(X_test), dtype=np.float32)
    total_iters = []
    importances = np.zeros(X.shape[1], dtype=np.float64)

    for seed in seeds:
        oof_s = np.zeros(len(X), dtype=np.float32)
        test_s = np.zeros(len(X_test), dtype=np.float32)
        for fold, (tr, va) in enumerate(splits, 1):
            print(f"--- LightGBM seed={seed} fold {fold}/{nfold} ---", flush=True)
            model = LGBMClassifier(**_lgbm_params(seed))
            model.fit(
                X[tr], y[tr],
                eval_set=[(X[va], y[va])],
                eval_metric="auc",
                callbacks=[early_stopping(EARLY_STOP, verbose=False)],
            )
            oof_s[va] = model.predict_proba(X[va])[:, 1]
            test_s += model.predict_proba(X_test)[:, 1] / nfold
            total_iters.append(model.best_iteration_)
            importances += np.asarray(
                model.feature_importances_, dtype=np.float64
            )
            del model
            gc.collect()
        oof_total += oof_s / len(seeds)
        test_total += test_s / len(seeds)
        del oof_s, test_s
        gc.collect()

    auc = roc_auc_score(y, oof_total)
    print(f"LightGBM multi-seed OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof_total, test_total, auc, total_iters, importances


def train_xgb_competitive(
    X, y, X_test, nfold=N_FOLDS, seeds=None, feature_names=None
):
    """XGBoost (hist) with multi-seed averaging. Returns same as lgbm."""
    seeds = seeds if seeds is not None else SEEDS
    skf = _make_cv(nfold)
    splits = list(skf.split(X, y))

    oof_total = np.zeros(len(X), dtype=np.float32)
    test_total = np.zeros(len(X_test), dtype=np.float32)
    total_iters = []
    importances = np.zeros(X.shape[1], dtype=np.float64)

    for seed in seeds:
        oof_s = np.zeros(len(X), dtype=np.float32)
        test_s = np.zeros(len(X_test), dtype=np.float32)
        for fold, (tr, va) in enumerate(splits, 1):
            print(f"--- XGBoost seed={seed} fold {fold}/{nfold} ---", flush=True)
            model = XGBClassifier(**_xgb_params(seed))
            model.fit(
                X[tr], y[tr],
                eval_set=[(X[va], y[va])],
                verbose=False,
            )
            oof_s[va] = model.predict_proba(X[va])[:, 1]
            test_s += model.predict_proba(X_test)[:, 1] / nfold
            total_iters.append(int(model.best_iteration))
            importances += np.asarray(
                model.feature_importances_, dtype=np.float64
            )
            del model
            gc.collect()
        oof_total += oof_s / len(seeds)
        test_total += test_s / len(seeds)
        del oof_s, test_s
        gc.collect()

    auc = roc_auc_score(y, oof_total)
    print(f"XGBoost multi-seed OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof_total, test_total, auc, total_iters, importances


def train_catboost_competitive(
    X, y, X_test, nfold=N_FOLDS, seeds=None, feature_names=None
):
    """CatBoost with multi-seed averaging. Returns same as lgbm.

    X may be a DataFrame with categorical string columns (passed as
    cat_features) or a numpy matrix.
    """
    seeds = seeds if seeds is not None else SEEDS
    skf = _make_cv(nfold)
    splits = list(skf.split(X, y))

    cat_features = None
    if isinstance(X, pd.DataFrame):
        cat_features = [
            c for c in X.columns if X[c].dtype == object
        ] or None
    n_feat = X.shape[1]

    oof_total = np.zeros(len(X), dtype=np.float32)
    test_total = np.zeros(len(X_test), dtype=np.float32)
    total_iters = []
    importances = np.zeros(n_feat, dtype=np.float64)

    for seed in seeds:
        oof_s = np.zeros(len(X), dtype=np.float32)
        test_s = np.zeros(len(X_test), dtype=np.float32)
        for fold, (tr, va) in enumerate(splits, 1):
            print(f"--- CatBoost seed={seed} fold {fold}/{nfold} ---", flush=True)
            model = CatBoostClassifier(**_cb_params(seed))
            fit_kwargs = {
                "eval_set": (X[va], y[va]),
                "early_stopping_rounds": EARLY_STOP,
                "verbose": False,
            }
            if cat_features is not None:
                fit_kwargs["cat_features"] = cat_features
            model.fit(X[tr], y[tr], **fit_kwargs)
            oof_s[va] = model.predict_proba(X[va])[:, 1]
            test_s += model.predict_proba(X_test)[:, 1] / nfold
            total_iters.append(int(model.get_best_iteration()))
            try:
                importances += np.asarray(
                    model.get_feature_importance(), dtype=np.float64
                )
            except Exception:
                pass
            del model
            gc.collect()
        oof_total += oof_s / len(seeds)
        test_total += test_s / len(seeds)
        del oof_s, test_s
        gc.collect()

    auc = roc_auc_score(y, oof_total)
    print(f"CatBoost multi-seed OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof_total, test_total, auc, total_iters, importances


# ---------------------------------------------------------------------------
# Legacy trainers (kept for run_pipeline.py compatibility)
# ---------------------------------------------------------------------------

def train_lgbm(X, y, X_test, nfold=N_FOLDS):
    """LightGBM baseline with early stopping on each fold's validation."""
    skf = _make_cv(nfold)
    oof = np.zeros(len(X), dtype=np.float32)
    test = np.zeros(len(X_test), dtype=np.float32)

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print(f"--- LightGBM fold {fold}/{nfold} ---", flush=True)
        model = LGBMClassifier(**_lgbm_params(SEED))
        model.fit(
            X[tr], y[tr],
            eval_set=[(X[va], y[va])],
            eval_metric="auc",
            callbacks=[early_stopping(200, verbose=False)],
        )
        oof[va] = model.predict_proba(X[va])[:, 1]
        test += model.predict_proba(X_test)[:, 1] / nfold
        del model
        gc.collect()

    auc = roc_auc_score(y, oof)
    print(f"LightGBM OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof, test, auc


def train_xgb(X, y, X_test, nfold=N_FOLDS):
    """XGBoost (hist) with early stopping."""
    skf = _make_cv(nfold)
    oof = np.zeros(len(X), dtype=np.float32)
    test = np.zeros(len(X_test), dtype=np.float32)

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print(f"--- XGBoost fold {fold}/{nfold} ---", flush=True)
        model = XGBClassifier(**_xgb_params(SEED))
        model.fit(
            X[tr], y[tr],
            eval_set=[(X[va], y[va])],
            verbose=False,
        )
        oof[va] = model.predict_proba(X[va])[:, 1]
        test += model.predict_proba(X_test)[:, 1] / nfold
        del model
        gc.collect()

    auc = roc_auc_score(y, oof)
    print(f"XGBoost OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof, test, auc


def train_catboost(X, y, X_test, nfold=N_FOLDS):
    """CatBoost baseline with early stopping on validation AUC."""
    skf = _make_cv(nfold)
    oof = np.zeros(len(X), dtype=np.float32)
    test = np.zeros(len(X_test), dtype=np.float32)

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print(f"--- CatBoost fold {fold}/{nfold} ---", flush=True)
        model = CatBoostClassifier(**_cb_params(SEED))
        model.fit(
            X[tr], y[tr],
            eval_set=(X[va], y[va]),
            early_stopping_rounds=200,
            verbose=False,
        )
        oof[va] = model.predict_proba(X[va])[:, 1]
        test += model.predict_proba(X_test)[:, 1] / nfold
        del model
        gc.collect()

    auc = roc_auc_score(y, oof)
    print(f"CatBoost OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof, test, auc


def train_lr(X, y, X_test, nfold=N_FOLDS):
    """Logistic Regression (lbfgs) with median imputation and scaling."""
    skf = _make_cv(nfold)
    oof = np.zeros(len(X), dtype=np.float32)
    test = np.zeros(len(X_test), dtype=np.float32)

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print(f"--- LogisticRegression fold {fold}/{nfold} ---", flush=True)
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        X_train = scaler.fit_transform(imputer.fit_transform(X[tr]))
        X_val = scaler.transform(imputer.transform(X[va]))
        X_test_imp = scaler.transform(imputer.transform(X_test))

        model = LogisticRegression(
            C=0.3,
            solver="lbfgs",
            max_iter=2000,
            random_state=SEED,
        )
        model.fit(X_train, y[tr])
        oof[va] = model.predict_proba(X_val)[:, 1]
        test += model.predict_proba(X_test_imp)[:, 1] / nfold
        del imputer, scaler, model, X_train, X_val, X_test_imp
        gc.collect()

    auc = roc_auc_score(y, oof)
    print(f"LogisticRegression OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof, test, auc


def train_et(X, y, X_test, nfold=N_FOLDS):
    """ExtraTrees as a diversity member of the ensemble."""
    skf = _make_cv(nfold)
    oof = np.zeros(len(X), dtype=np.float32)
    test = np.zeros(len(X_test), dtype=np.float32)

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print(f"--- ExtraTrees fold {fold}/{nfold} ---", flush=True)
        model = ExtraTreesClassifier(
            n_estimators=400,
            max_features=0.5,
            min_samples_leaf=10,
            n_jobs=N_JOBS,
            random_state=SEED,
        )
        model.fit(X[tr], y[tr])
        oof[va] = model.predict_proba(X[va])[:, 1]
        test += model.predict_proba(X_test)[:, 1] / nfold
        del model
        gc.collect()

    auc = roc_auc_score(y, oof)
    print(f"ExtraTrees OOF ROC-AUC: {auc:.5f}", flush=True)
    return oof, test, auc


def train_all(X, y, X_test, nfold=N_FOLDS):
    """
    Train the full model lineup and return dicts of predictions.

    Returns
    -------
    (oofs, tests, aucs) : dict[str, np.ndarray] x3
        Keys: lgbm, xgb, catboost, lr, et
    """
    oofs, tests, aucs = {}, {}, {}

    for name, fn in [
        ("lgbm", train_lgbm),
        ("xgb", train_xgb),
        ("catboost", train_catboost),
        ("lr", train_lr),
        ("et", train_et),
    ]:
        print(f"\n[{name}] starting...", flush=True)
        oofs[name], tests[name], aucs[name] = fn(X, y, X_test, nfold=nfold)

    return oofs, tests, aucs


def simple_average(oof_preds, test_preds, y):
    """Simple average of model predictions (helper function for pipelines)."""
    blended_oof = np.mean(oof_preds, axis=0)
    blended_test = np.mean(test_preds, axis=0)
    auc = roc_auc_score(y, blended_oof)
    return auc, blended_test


## Módulo: `src/ensemble.py`

In [ ]:
"""
Ensemble methods for Home Credit Default Risk competition.
==========================================================

Competitive sprint methods (GBDT-only):
1. Probabilistic weighted blend - scipy Nelder-Mead over 3 weights
   (sum=1, w>=0) on calibrated OOF probabilities
2. Rank-averaging blend - scipy Nelder-Mead over 3 weights on normalized
   ranks (robust to model overconfidence)
3. Simple average (equal weights) - sanity fallback

The pipeline keeps whichever of (1) or (2) maximizes 5-fold OOF ROC-AUC.
Legacy methods (rank_ensemble, stacked_ensemble) are kept for
run_pipeline.py compatibility.

Public entry point:
    run_competitive_pipeline(train_path, test_path, output_path)
"""

import gc
import os

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

import features as _features_mod
import models as _models_mod


def _nelder_mead_blend(oof_stack, test_stack, y, method="prob"):
    """
    Optimize 3 non-negative weights (sum constrained by normalization)
    with Nelder-Mead. The objective normalizes weights internally so the
    unconstrained optimizer explores the simplex.

    method="prob"  : weights on calibrated probabilities
    method="rank"  : weights on normalized ranks
    """
    n = oof_stack.shape[1]

    def transform(stack):
        if method == "rank":
            out = np.empty_like(stack, dtype=np.float64)
            for j in range(stack.shape[1]):
                out[:, j] = np.argsort(np.argsort(stack[:, j])) / stack.shape[0]
            return out
        return stack

    oof_t = transform(oof_stack).astype(np.float64)
    test_t = transform(test_stack).astype(np.float64)

    def objective(w):
        s = np.abs(w).sum()
        if s <= 0:
            return 1.0
        wn = np.abs(w) / s
        blended = oof_t @ wn
        return -roc_auc_score(y, blended)

    start = np.ones(n) / n
    result = minimize(
        objective, start, method="Nelder-Mead",
        options={"maxiter": 500, "xatol": 1e-8, "fatol": 1e-9},
    )
    w = np.abs(result.x)
    w = w / w.sum()
    blended_oof = oof_t @ w
    blended_test = test_t @ w
    auc = roc_auc_score(y, blended_oof)
    return w, auc, blended_test


def gbdt_ensemble(oof_preds, test_preds, y, names=("lgbm", "catboost", "xgb")):
    """
    GBDT-only optimal ensemble.

    Tries (a) probabilistic Nelder-Mead blend and (b) rank Nelder-Mead
    blend, plus (c) simple average as a sanity baseline. Returns the
    method with the best OOF ROC-AUC.

    Parameters
    ----------
    oof_preds : list of np.ndarray (3)
    test_preds : list of np.ndarray (3)
    y : np.ndarray

    Returns
    -------
    (best_name, best_auc, weights, best_test, all_results)
    all_results: dict name -> (auc, weights)
    """
    oof_stack = np.column_stack(oof_preds)
    test_stack = np.column_stack(test_preds)

    w_prob, auc_prob, blend_prob = _nelder_mead_blend(
        oof_stack, test_stack, y, method="prob"
    )
    w_rank, auc_rank, blend_rank = _nelder_mead_blend(
        oof_stack, test_stack, y, method="rank"
    )
    simple_test = test_stack.mean(axis=1)
    auc_simple = roc_auc_score(y, oof_stack.mean(axis=1))

    results = {
        "prob_blend": (auc_prob, w_prob),
        "rank_blend": (auc_rank, w_rank),
        "simple_avg": (auc_simple, np.ones(3) / 3),
    }

    best = max(results, key=lambda k: results[k][0])
    if best == "prob_blend":
        return best, auc_prob, w_prob, blend_prob, results
    if best == "rank_blend":
        return best, auc_rank, w_rank, blend_rank, results
    return best, auc_simple, np.ones(3) / 3, simple_test, results


def rank_ensemble(oof_preds, test_preds, y):
    """Legacy equal-weight rank ensemble (kept for run_pipeline.py)."""
    oof_ranks, test_ranks = [], []
    for arr in oof_preds:
        oof_ranks.append(np.argsort(np.argsort(arr)).astype(np.float32) / len(arr))
    for arr in test_preds:
        test_ranks.append(np.argsort(np.argsort(arr)).astype(np.float32) / len(arr))
    blended_oof = np.mean(oof_ranks, axis=0)
    blended_test = np.mean(test_ranks, axis=0)
    auc = roc_auc_score(y, blended_oof)
    return auc, blended_test


def simple_average(oof_preds, test_preds, y):
    """Simple average of model predictions."""
    blended_oof = np.mean(oof_preds, axis=0)
    blended_test = np.mean(test_preds, axis=0)
    auc = roc_auc_score(y, blended_oof)
    return auc, blended_test


def weighted_blend(oof_preds, test_preds, y):
    """Legacy SLSQP weighted blend (kept for run_pipeline.py)."""
    n_models = len(oof_preds)

    def objective(w):
        blended = np.sum(w * np.column_stack(oof_preds), axis=1)
        return -roc_auc_score(y, blended)

    start_w = np.ones(n_models) / n_models
    bounds = [(0, 1) for _ in range(n_models)]
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
    result = minimize(
        objective, start_w, method="SLSQP",
        bounds=bounds, constraints=constraints,
        options={"ftol": 1e-9, "maxiter": 100},
    )
    best_weights = result.x
    best_auc = -result.fun
    blended_test = np.sum(best_weights * np.column_stack(test_preds), axis=1)
    return best_weights, best_auc, blended_test


def stacked_ensemble(oof_preds, test_preds, y):
    """Legacy logistic-regression stacking meta-model (kept for run_pipeline.py)."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    X_meta = np.column_stack(oof_preds)
    scaler = StandardScaler()
    X_meta_scaled = scaler.fit_transform(X_meta)

    meta_model = LogisticRegression(
        max_iter=1000, C=0.1, solver="lbfgs", random_state=42
    )
    meta_model.fit(X_meta_scaled, y)

    blended_oof = meta_model.predict_proba(X_meta_scaled)[:, 1]
    X_test_meta = np.column_stack(test_preds)
    X_test_meta_scaled = scaler.transform(X_test_meta)
    blended_test = meta_model.predict_proba(X_test_meta_scaled)[:, 1]

    auc = roc_auc_score(y, blended_oof)
    return auc, blended_test, meta_model


def create_submission(test_preds, filename="submission.csv", row_ids=None):
    """
    Create submission file in the expected format.

    Parameters
    ----------
    test_preds : np.ndarray
        Predictions for all rows in test set (must match sample_submission length)
    filename : str
        Output file name
    row_ids : pd.Series or None
        SK_ID_CURR values from test set; if None, uses sample submission index

    Returns
    -------
    pd.DataFrame
    """
    sample_path = "D:/Usuario/orca/CGI-World/Competition/input/sample_submission.csv"
    if not os.path.exists(sample_path):
        sample_path = "Competition/input/sample_submission.csv"
    submission = pd.read_csv(sample_path)
    submission["TARGET"] = test_preds[: len(submission)].astype("float32")
    out_dir = "D:/Usuario/orca/CGI-World/output"
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, filename)
    submission.to_csv(out_path, index=False)
    return submission


# ---------------------------------------------------------------------------
# Competitive pipeline entry point
# ---------------------------------------------------------------------------

def run_competitive_pipeline(
    train_path=None,
    test_path=None,
    output_path=None,
    seeds=None,
    prune=True,
):
    """
    Full competitive sprint pipeline:

    1. Load + sort by SK_ID_CURR
    2. build_features (leak-free)
    3. OOF smoothed target encoding (m=20) computed strictly inside folds
    4. Train 3 GBDT families x multi-seed, average per family
    5. Optional zero-importance pruning (only if it improves CV AUC)
    6. GBDT-only ensemble: Nelder-Mead prob blend vs rank blend
    7. Write submission

    Returns dict with aucs, best ensemble, weights and feature stats.
    """
    base_dir = "D:/Usuario/orca/CGI-World"
    if train_path is None:
        train_path = os.path.join(base_dir, "Competition/input/train.csv")
    if test_path is None:
        test_path = os.path.join(base_dir, "Competition/input/test.csv")
    if output_path is None:
        output_path = os.path.join(base_dir, "output/submission.csv")
    seeds = seeds if seeds is not None else _models_mod.SEEDS
    N_JOBS_OK = True  # noqa: F841 (n_jobs enforced inside model params)

    print("=" * 60, flush=True)
    print("COMPETITIVE PIPELINE - Home Credit Default Risk", flush=True)
    print("=" * 60, flush=True)

    print("[1] Loading data...", flush=True)
    train_df = pd.read_csv(train_path).sort_values("SK_ID_CURR")
    test_df = pd.read_csv(test_path).sort_values("SK_ID_CURR")
    for df in (train_df, test_df):
        for col in df.select_dtypes(include=["float64"]).columns:
            df[col] = df[col].astype("float32")
        for col in df.select_dtypes(include=["int64"]).columns:
            df[col] = df[col].astype("int32")

    print("[2] Building features...", flush=True)
    train_proc, test_proc = _features_mod.build_features(train_df, test_df)
    n_feat_raw = train_proc.shape[1] - 2  # minus TARGET and SK_ID_CURR
    print(f"Features: {n_feat_raw}, Train: {len(train_proc)}, Test: {len(test_proc)}", flush=True)

    y = train_proc["TARGET"].values
    X = train_proc.drop(columns=["TARGET", "SK_ID_CURR"]).astype(np.float32)
    X_test = test_proc.drop(columns=["SK_ID_CURR"]).astype(np.float32)
    feat_cols = list(X.columns)
    test_ids = test_proc["SK_ID_CURR"].values.astype("int32")
    del train_proc, test_proc
    gc.collect()

    # --- 3. Out-of-fold smoothed target encoding (m=20, strict per fold) ---
    print("[3] Out-of-fold smoothed target encoding (m=20)...", flush=True)
    train_cats, test_cats, cat_cols = _features_mod.get_target_encoding_cats(
        train_df,
        test_df,
        cols=_features_mod.TARGET_ENCODED_COLS,
    )
    # align with X row order
    assert len(train_cats) == len(y) and len(test_cats) == len(X_test)

    skf = _models_mod._make_cv(_models_mod.N_FOLDS)
    oof_te = np.zeros((len(X), len(cat_cols)), dtype=np.float32)
    test_te = np.zeros((len(X_test), len(cat_cols)), dtype=np.float32)

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        print(f"--- TE fold {fold}/{_models_mod.N_FOLDS} ---", flush=True)
        for j, col in enumerate(cat_cols):
            fit_vals = train_cats[col].to_numpy()[tr]
            val_vals = train_cats[col].to_numpy()[va]
            te_vals = _features_mod.transform_target_encoding(
                *(_features_mod.fit_smoothed_target_encoding(y[tr], fit_vals, smoothing=20)),
                val_vals,
                smoothing=20,
            )
            oof_te[va, j] = te_vals
            te_test = _features_mod.transform_target_encoding(
                *(_features_mod.fit_smoothed_target_encoding(y[tr], fit_vals, smoothing=20)),
                test_cats[col].to_numpy(),
                smoothing=20,
            )
            test_te[:, j] += te_test / _models_mod.N_FOLDS
    del train_cats, test_cats
    gc.collect()

    te_names = [f"TE_{c}" for c in cat_cols]
    X = np.column_stack([X, oof_te]).astype(np.float32)
    X_test = np.column_stack([X_test, test_te]).astype(np.float32)
    feat_cols = feat_cols + te_names
    print(f"Features after TE: {X.shape[1]}", flush=True)

    # --- 4. Train 3 GBDT families, multi-seed -------------------------------
    print("\n[4] Training competitive GBDTs (multi-seed)...", flush=True)
    oof_lgbm, test_lgbm, auc_lgbm, iters_lgbm, imp_lgbm = (
        _models_mod.train_lgbm_competitive(X, y, X_test, seeds=seeds)
    )
    oof_xgb, test_xgb, auc_xgb, iters_xgb, imp_xgb = (
        _models_mod.train_xgb_competitive(X, y, X_test, seeds=seeds)
    )
    oof_cb, test_cb, auc_cb, iters_cb, imp_cb = (
        _models_mod.train_catboost_competitive(X, y, X_test, seeds=seeds)
    )

    print("\n[5] Single-seed (42) baseline family AUCs:", flush=True)
    print(f"  LightGBM: {auc_lgbm:.5f}", flush=True)
    print(f"  XGBoost:  {auc_xgb:.5f}", flush=True)
    print(f"  CatBoost: {auc_cb:.5f}", flush=True)

    # --- 5. Optional zero-importance pruning ---------------------------------
    pruned = False
    if prune:
        print("\n[5b] Zero-importance pruning check...", flush=True)
        total_imp = imp_lgbm + imp_xgb + imp_cb
        keep = total_imp > 0
        n_dropped = int((~keep).sum())
        if 0 < n_dropped < X.shape[1]:
            X_p, X_test_p = X[:, keep], X_test[:, keep]
            kept_names = [c for c, k in zip(feat_cols, keep) if k]
            print(f"  Dropping {n_dropped} zero-importance features -> {X_p.shape[1]}", flush=True)
            # quick single-seed validation on pruned set
            oof_l_p, _, auc_l_p, _, _ = _models_mod.train_lgbm_competitive(
                X_p, y, X_test_p, seeds=[42]
            )
            oof_x_p, _, auc_x_p, _, _ = _models_mod.train_xgb_competitive(
                X_p, y, X_test_p, seeds=[42]
            )
            oof_c_p, _, auc_c_p, _, _ = _models_mod.train_catboost_competitive(
                X_p, y, X_test_p, seeds=[42]
            )
            best_before = max(auc_lgbm, auc_xgb, auc_cb)
            best_after = max(auc_l_p, auc_x_p, auc_c_p)
            if best_after >= best_before:
                X, X_test = X_p, X_test_p
                feat_cols = kept_names
                pruned = True
                print(f"  Pruned set OK (best single-seed {best_after:.5f} >= {best_before:.5f}), keeping pruned.", flush=True)
            else:
                print(f"  Pruned set worse ({best_after:.5f} < {best_before:.5f}), keeping full set.", flush=True)
                del X_p, X_test_p
            del oof_l_p, auc_l_p, oof_x_p, auc_x_p, oof_c_p, auc_c_p
            gc.collect()
        else:
            print("  No zero-importance features to drop.", flush=True)

    # --- 6. GBDT-only ensemble ----------------------------------------------
    print("\n[6] GBDT-only ensemble optimization...", flush=True)
    best_name, best_auc, weights, best_test, all_results = gbdt_ensemble(
        [oof_lgbm, oof_cb, oof_xgb],
        [test_lgbm, test_cb, test_xgb],
        y,
    )
    print(f"  Prob blend : {all_results['prob_blend'][0]:.5f}  w={np.round(all_results['prob_blend'][1], 4)}", flush=True)
    print(f"  Rank blend : {all_results['rank_blend'][0]:.5f}  w={np.round(all_results['rank_blend'][1], 4)}", flush=True)
    print(f"  Simple avg : {all_results['simple_avg'][0]:.5f}", flush=True)
    print(f"  BEST: {best_name} with OOF AUC {best_auc:.5f}", flush=True)

    # --- 7. Submission --------------------------------------------------------
    print("\n[7] Writing submission...", flush=True)
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    sample_path = os.path.join(base_dir, "Competition/input/sample_submission.csv")
    if os.path.exists(sample_path):
        submission = pd.read_csv(sample_path)
        submission["TARGET"] = best_test[: len(X_test)].astype(np.float32)
    else:
        submission = pd.DataFrame(
            {
                "SK_ID_CURR": test_ids,
                "TARGET": best_test[: len(X_test)].astype(np.float32),
            }
        )
    submission.to_csv(output_path, index=False)
    print(f"Saved to: {output_path} ({len(submission)} rows)", flush=True)

    summary = {
        "n_features": int(X.shape[1]),
        "pruned": pruned,
        "lgbm": float(auc_lgbm),
        "xgb": float(auc_xgb),
        "catboost": float(auc_cb),
        "ensemble_method": best_name,
        "ensemble_weights": {
            "lgbm": float(weights[0]),
            "catboost": float(weights[1]),
            "xgb": float(weights[2]),
        },
        "ensemble_auc": float(best_auc),
        "all_ensembles": {k: float(v[0]) for k, v in all_results.items()},
        "seeds": list(seeds),
        "lgbm_iters": iters_lgbm,
        "xgb_iters": iters_xgb,
        "catboost_iters": iters_cb,
    }
    print("\n" + "=" * 60, flush=True)
    print("PIPELINE COMPLETE", flush=True)
    print("=" * 60, flush=True)
    return summary


## Módulo: `run_pipeline.py`

In [ ]:
#!/usr/bin/env python
"""
Pipeline de entrada para Home Credit Default Risk - GCI Global 2026.
======================================================
Resuelve output/submission.csv de la mejor fuente disponible (fail-fast):

  1. Si src/features.py + src/models.py + src/ensemble.py existen y expone el
     contrato de OpenCode 2 (build_features() -> (X_train, y, X_test),
     run_models() -> dict nombre->oof/funcion de prediccion,
     ensemble() -> probs_test), se usa el pipeline avanzado.
  2. En caso contrario, ejecuta src/baseline.py (misma semilla, mismo CV)
     para producir una submission validada.

Reproducibilidad: seed=42, CV StratifiedKFold(n_splits=5, shuffle=True,
random_state=42) sobre train.csv ordenado ascendentemente por SK_ID_CURR.
"""
import subprocess
import sys
import traceback
from pathlib import Path

ROOT = Path(__file__).resolve().parent
SRC = ROOT / 'src'


def advanced_pipeline_available():
    for mod in ('features.py', 'models.py', 'ensemble.py'):
        if not (SRC / mod).exists():
            return False
    try:
        sys.path.insert(0, str(SRC))
        import features, models, ensemble  # noqa: F401
        for obj in ('build_features', 'run_models', 'ensemble_predictions'):
            if not (hasattr(features, obj) or hasattr(models, obj) or hasattr(ensemble, obj)):
                # el objeto puede estar en cualquier modulo; se resuelve abajo
                pass
        return True
    except Exception as e:
        print(f'[pipeline] Modulos de OpenCode 2 presentes pero con error al importar: {e}')
        return False


def run_advanced_pipeline():
    sys.path.insert(0, str(SRC))
    import features
    import models
    import ensemble
    import pandas as pd

    train = pd.read_csv(ROOT / 'Competition' / 'input' / 'train.csv')
    test = pd.read_csv(ROOT / 'Competition' / 'input' / 'test.csv')
    train = train.sort_values('SK_ID_CURR').reset_index(drop=True)
    test = test.sort_values('SK_ID_CURR').reset_index(drop=True)

    X_train, y, X_test = features.build_features(train, test)
    preds = models.run_models(X_train, y, X_test)  # dict {name: pred_test o (oof, pred_test)}
    proba = ensemble.ensemble_predictions(preds)   # array-like (len(test),)

    sub = pd.DataFrame({
        'SK_ID_CURR': test['SK_ID_CURR'].values,
        'TARGET': np.float64(proba),
    })
    # Orden exacto de sample_submission.csv
    ref = pd.read_csv(ROOT / 'Competition' / 'input' / 'sample_submission.csv')
    sub = sub.set_index('SK_ID_CURR').reindex(ref['SK_ID_CURR'].values).reset_index()
    out = ROOT / 'output' / 'submission.csv'
    out.parent.mkdir(exist_ok=True)
    sub.to_csv(out, index=False)
    print(f'[pipeline] output/submission.csv generado por pipeline avanzado: {len(sub)} filas')


def run_baseline_fallback():
    print('[pipeline] Modulos de OpenCode 2 ausentes; ejecuto src/baseline.py (fallback).')
    r = subprocess.run([sys.executable, str(SRC / 'baseline.py')], cwd=ROOT)
    if r.returncode != 0:
        raise SystemExit(f'[pipeline] baseline.py fallo con codigo {r.returncode}')


if __name__ == '__main__':
    try:
        if advanced_pipeline_available():
            run_advanced_pipeline()
        else:
            run_baseline_fallback()
    except Exception:
        traceback.print_exc()
        if not (ROOT / 'output' / 'submission.csv').exists():
            raise
        print('[pipeline] Excepcion en pipeline avanzado; se conserva el submission existente si hay.')


In [ ]:
# Generación de submission.npy y model.pt
import numpy as np, pandas as pd, torch
sub = pd.read_csv('output/submission.csv')
np.save('submission.npy', sub['TARGET'].to_numpy(dtype=np.float32))
